In [18]:
import os
import pandas as pd
from ucimlrepo import fetch_ucirepo

data = fetch_ucirepo(id=967)
df = data.data.original[["URL", "label"]]

os.makedirs("data", exist_ok=True)
df.to_csv("data/phishing_urls.csv", index=False)
print("Saved", len(df), "URLs")

Saved 235795 URLs


In [19]:
print(df["label"].value_counts())
df.sample(10, random_state=1)

label
1    134850
0    100945
Name: count, dtype: int64


,URL,label
71259,https://www.dkom.hr,1
39785,https://www.loomlove.com,1
43857,https://www.indianbureaucracy.com,1
194428,http://ethereum-uma.com,0
52413,https://www.alephbookcompany.com,1
32513,https://www.handisport.org,1
188432,https://www.cavernas.org.br,1
115394,https://www.rewview.co.kr,1
8887,https://www.thechronicle.com,1
221925,https://www.blackpointt.com,0


In [20]:
df = df.copy()
df["is_phishing"] = (df["label"] == 0).astype(int)
df["is_phishing"].value_counts()

is_phishing
0    134850
1    100945
Name: count, dtype: int64

In [21]:
import re

def url_features(url):
    return {
        "length": len(url),
        "num_dots": url.count("."),
        "num_hyphens": url.count("-"),
        "num_digits": sum(c.isdigit() for c in url),
        "num_slashes": url.count("/"),
        "has_at": int("@" in url),
        "has_ip": int(bool(re.search(r"\d+\.\d+\.\d+\.\d+", url))),
        "is_https": int(url.lower().startswith("https")),
    }

X = pd.DataFrame([url_features(u) for u in df["URL"]])
y = df["is_phishing"]
X.head()

,length,num_dots,num_hyphens,num_digits,num_slashes,has_at,has_ip,is_https
0,32,2,0,0,2,0,0,1
1,24,2,1,0,2,0,0,1
2,30,3,0,0,2,0,0,1
3,27,2,0,0,2,0,0,1
4,34,2,0,0,2,0,0,1


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=["legit", "phishing"]))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

       legit       0.99      1.00      0.99     26970
    phishing       1.00      0.98      0.99     20189

    accuracy                           0.99     47159
   macro avg       0.99      0.99      0.99     47159
weighted avg       0.99      0.99      0.99     47159

[[26961     9]
 [  308 19881]]


In [23]:
tests = [
    "https://www.google.com",
    "http://paypal-login-secure.verify-account.xyz/signin",
    "http://192.168.10.5/bank/update.php",
]

for u in tests:
    p = model.predict_proba(pd.DataFrame([url_features(u)]))[0, 1]
    print(f"{p:.0%} phishing   {u}")

0% phishing   https://www.google.com
100% phishing   http://paypal-login-secure.verify-account.xyz/signin
100% phishing   http://192.168.10.5/bank/update.php


In [24]:
coefs = pd.Series(model[-1].coef_[0], index=X.columns).sort_values()
coefs

is_https       -6.194683
has_ip         -0.377247
num_dots       -0.137956
has_at          0.429332
num_hyphens     1.436419
length          5.637915
num_digits      9.845995
num_slashes    11.964560
dtype: float64

In [25]:
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

Using: mps


In [26]:
MAX_LEN = 200

def encode(url):
    codes = [min(ord(c), 127) for c in url[:MAX_LEN]]
    return codes + [0] * (MAX_LEN - len(codes))

print(encode("http://abc.com")[:20])

[104, 116, 116, 112, 58, 47, 47, 97, 98, 99, 46, 99, 111, 109, 0, 0, 0, 0, 0, 0]


In [27]:
train_urls = df.loc[X_train.index, "URL"]
test_urls = df.loc[X_test.index, "URL"]

Xtr = torch.tensor([encode(u) for u in train_urls])
Xte = torch.tensor([encode(u) for u in test_urls])
ytr = torch.tensor(y_train.values, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=512, shuffle=True)
print(Xtr.shape, Xte.shape)

torch.Size([188636, 200]) torch.Size([47159, 200])


In [28]:
class URLNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(128, 32)
        self.conv = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embed(x)           # each character -> 32 numbers
        x = x.transpose(1, 2)       # reshape for the next layer
        x = torch.relu(self.conv(x))
        x = x.max(dim=2).values     # keep the strongest signal from each pattern detector
        return self.out(x).squeeze(1)

model_nn = URLNet().to(device)
model_nn

URLNet(
  (embed): Embedding(128, 32)
  (conv): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (out): Linear(in_features=64, out_features=1, bias=True)
)

In [29]:
opt = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

start = time.perf_counter()
for epoch in range(3):
    model_nn.train()
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model_nn(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1}: loss {total/len(train_loader):.4f}")
print(f"Training took {time.perf_counter() - start:.0f} seconds")

Epoch 1: loss 0.0611
Epoch 2: loss 0.0132
Epoch 3: loss 0.0121
Training took 7 seconds


In [30]:
model_nn.eval()
probs = []
with torch.no_grad():
    for i in range(0, len(Xte), 2048):
        probs.append(torch.sigmoid(model_nn(Xte[i:i+2048].to(device))).cpu())

pred_nn = (torch.cat(probs) > 0.5).int().numpy()
print(classification_report(y_test, pred_nn, target_names=["legit", "phishing"]))
print(confusion_matrix(y_test, pred_nn))

              precision    recall  f1-score   support

       legit       1.00      1.00      1.00     26970
    phishing       1.00      1.00      1.00     20189

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

[[26970     0]
 [   92 20097]]


In [31]:
def nn_score(url):
    x = torch.tensor([encode(url)]).to(device)
    with torch.no_grad():
        return torch.sigmoid(model_nn(x)).item()

for u in tests:
    base = model.predict_proba(pd.DataFrame([url_features(u)]))[0, 1]
    print(f"baseline {base:.0%} | neural net {nn_score(u):.0%} | {u}")

baseline 0% | neural net 0% | https://www.google.com
baseline 100% | neural net 100% | http://paypal-login-secure.verify-account.xyz/signin
baseline 100% | neural net 100% | http://192.168.10.5/bank/update.php


In [32]:
bare = df["URL"].str.replace(r"^https?://", "", regex=True)
check = pd.DataFrame({
    "is_phishing": df["is_phishing"],
    "uses_https": df["URL"].str.startswith("https"),
    "has_path": bare.str.contains(r"/."),
})
check.groupby("is_phishing").mean().round(3)

,uses_https,has_path
is_phishing,,
0,1.000,0.000
1,0.487,0.284


In [33]:
real_legit = [
    "https://docs.python.org/3/library/re.html",
    "https://github.com/pytorch/pytorch/issues/12345",
    "https://www.bbc.co.uk/news/technology",
    "https://www.gov.uk/check-uk-visa/y",
    "http://example.com",
]

for u in real_legit:
    base = model.predict_proba(pd.DataFrame([url_features(u)]))[0, 1]
    print(f"baseline {base:.0%} | neural net {nn_score(u):.0%} | {u}")

baseline 100% | neural net 100% | https://docs.python.org/3/library/re.html
baseline 100% | neural net 100% | https://github.com/pytorch/pytorch/issues/12345
baseline 100% | neural net 86% | https://www.bbc.co.uk/news/technology
baseline 100% | neural net 98% | https://www.gov.uk/check-uk-visa/y
baseline 100% | neural net 100% | http://example.com


In [34]:
from urllib.parse import urlsplit

def get_host(url):
    try:
        host = urlsplit(url if "://" in url else "http://" + url).hostname or ""
    except ValueError:
        host = ""
    return host.removeprefix("www.")

hosts = pd.DataFrame({"host": df["URL"].map(get_host), "is_phishing": df["is_phishing"]})
hosts = hosts[hosts["host"] != ""].drop_duplicates("host")
print(len(hosts), "unique domains")
hosts.groupby("is_phishing").sample(5, random_state=1)

219094 unique domains


,host,is_phishing
70997,eurogroupforanimals.org,0
207974,jlsoftarts.com,0
172832,donqinn.net,0
5226,mospi.gov.in,0
233417,sunnysafaris.com,0
175748,335166.com,1
83888,neighborly-zippy-wildebeest.glitch.me,1
36677,marjol0k5on.duckdns.org,1
147279,en.mcafee-service.us.com,1
24446,webmail-106627.weeblysite.com,1


In [38]:
h_train, h_test = train_test_split(
    hosts, test_size=0.2, random_state=42, stratify=hosts["is_phishing"]
)

Xtr_h = torch.tensor([encode(h) for h in h_train["host"]])
Xte_h = torch.tensor([encode(h) for h in h_test["host"]])
ytr_h = torch.tensor(h_train["is_phishing"].values, dtype=torch.float32)
loader_h = DataLoader(TensorDataset(Xtr_h, ytr_h), batch_size=512, shuffle=True)

model_h = URLNet().to(device)
opt = torch.optim.Adam(model_h.parameters(), lr=1e-3)

for epoch in range(5):
    model_h.train()
    total = 0
    for xb, yb in loader_h:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model_h(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1}: loss {total/len(loader_h):.4f}")

Epoch 1: loss 0.4021
Epoch 2: loss 0.3395
Epoch 3: loss 0.3278
Epoch 4: loss 0.3216
Epoch 5: loss 0.3164


In [39]:
model_h.eval()
probs = []
with torch.no_grad():
    for i in range(0, len(Xte_h), 2048):
        probs.append(torch.sigmoid(model_h(Xte_h[i:i+2048].to(device))).cpu())

pred_h = (torch.cat(probs) > 0.5).int().numpy()
print(classification_report(h_test["is_phishing"], pred_h, target_names=["legit", "phishing"]))
print(confusion_matrix(h_test["is_phishing"], pred_h))

              precision    recall  f1-score   support

       legit       0.85      0.97      0.90     26963
    phishing       0.93      0.72      0.81     16856

    accuracy                           0.87     43819
   macro avg       0.89      0.84      0.86     43819
weighted avg       0.88      0.87      0.87     43819

[[26069   894]
 [ 4721 12135]]


In [40]:
def host_score(url):
    x = torch.tensor([encode(get_host(url))]).to(device)
    with torch.no_grad():
        return torch.sigmoid(model_h(x)).item()

for u in real_legit + tests:
    print(f"{host_score(u):.0%} phishing | {get_host(u):35} | {u}")

74% phishing | docs.python.org                     | https://docs.python.org/3/library/re.html
28% phishing | github.com                          | https://github.com/pytorch/pytorch/issues/12345
5% phishing | bbc.co.uk                           | https://www.bbc.co.uk/news/technology
1% phishing | gov.uk                              | https://www.gov.uk/check-uk-visa/y
12% phishing | example.com                         | http://example.com
16% phishing | google.com                          | https://www.google.com
100% phishing | paypal-login-secure.verify-account.xyz | http://paypal-login-secure.verify-account.xyz/signin
99% phishing | 192.168.10.5                        | http://192.168.10.5/bank/update.php


In [47]:
for t in [0.5, 0.4, 0.3]:
    p = (torch.cat(probs) > t).int().numpy()
    cm = confusion_matrix(h_test["is_phishing"], p)
    print(f"threshold {t}: missed phishing {cm[1,0]}, false alarms {cm[0,1]}")

threshold 0.5: missed phishing 4721, false alarms 894
threshold 0.4: missed phishing 4160, false alarms 1726
threshold 0.3: missed phishing 3325, false alarms 3538


In [48]:
import ollama

def explain(url, llm="qwen3:8b"):
    score = host_score(url)
    prompt = f"""You are a SOC analyst. A machine-learning model scored this URL's domain as {score:.0%} likely phishing.

URL: {url}
Domain: {get_host(url)}

Based only on the text of the URL (you cannot visit it), write:
1. Verdict: Likely phishing / Suspicious / Likely safe
2. Two or three specific reasons, pointing at parts of the URL
3. One recommended action for the analyst
Keep it under 100 words."""
    reply = ollama.chat(model=llm, messages=[{"role": "user", "content": prompt}])
    return score, reply.message.content

In [49]:
queue = [
    "http://paypal-login-secure.verify-account.xyz/signin",
    "https://www.bbc.co.uk/news/technology",
    "http://amaz0n-account-update.info/verify",
    "https://microsoft365-security-alert.co/login",
    "https://github.com/pytorch/pytorch",
]

for u in queue:
    score = host_score(u)
    print("=" * 70)
    print(f"{u}\nModel score: {score:.0%}")
    if score >= 0.5:
        _, note = explain(u)
        print(note)
    else:
        print("Below threshold - no alert raised.")

http://paypal-login-secure.verify-account.xyz/signin
Model score: 100%
**Verdict:** Likely phishing.  
**Reasons:** 1. The domain "verify-account.xyz" is suspiciously generic and not a known PayPal domain. 2. The subdomain "paypal-login-secure" mimics PayPal’s structure but uses an untrusted TLD (.xyz). 3. The inclusion of "secure" is often used in phishing to mimic legitimate URLs.  
**Action:** Block the domain and flag for further analysis to prevent potential credential theft.
https://www.bbc.co.uk/news/technology
Model score: 5%
Below threshold - no alert raised.
http://amaz0n-account-update.info/verify
Model score: 98%
**Verdict:** Likely phishing  
**Reasons:** 1. Domain "amaz0n-account-update.info" contains a misspelled "amaz0n" (correct: "amazon"). 2. The .info TLD is often used for phishing. 3. The subdomain "amaz0n-account-update" mimics a fake service name.  
**Action:** Block the domain and monitor for similar domains. Check if the domain is registered to a suspicious enti